In [1]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import random
import numpy as np
import unidecode
import matplotlib.pyplot as plt
import plotly.graph_objects as go 
import plotly.express as px

In [17]:
df_gend = pd.read_csv("./gendarmerie/gendarmerie_geocoded.csv", sep=";",encoding_errors='ignore')


In [23]:
col_to_keep = ["identifiant_public_unite","adresse"]
df = df_gend[col_to_keep]
df.rename(columns={'adresse':'ADRESS'}, inplace=True)

df.dropna(subset="ADRESS")

df["ADRESS"]= df["ADRESS"].str.upper()

/tmp/ipykernel_2478178/3752102720.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'adresse':'ADRESS'}, inplace=True)
/tmp/ipykernel_2478178/3752102720.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["ADRESS"]= df["ADRESS"].str.upper()


In [24]:
voirie = ["RUE","COURS","COUR","VOIE","RUELLE","ESPLANADE",
          "PLACE","SQUARE","SQ","ROND POINT","PL",
          "IMPASSE", "ALLEE", "CHEMIN" ,"ROUTE","RTE","IMP","PROMENADE","ALLE","ALL",
          "AVENUE", "BOULEVARD","BD","AVE","BLD","BVD","BLV","AVN","BV",
          "FERME","DOMAINE","LIEU DIT","QUARTIER","QUR"]

##ATT AVEC COUR : A LA FOIS DANS LES ADRESSES ET COMME VOIRIE   

numeros = ["1","2","3","4","5","6","7","8","9","0"]

bruit = [ "RESIDENCE","CHEZ","HOPITAL","SDF","RES","MME","BAT","MAISON","MR","HOTEL",
         "LOTISSEMENT","CENTRE","QUARTIER","APPT","APT","SANTE", "RETRAITE", "HOP","TRANSFERT"]

In [28]:
def find_attribute(attributes, text, is_number=False):
    if pd.notna(text):
        if is_number:
            numbers = re.findall(r'\d+', text)
            
            return ','.join(numbers) if numbers else ""
        else:
            found_attributes = [x for x in attributes if re.search(r'\b' + re.escape(x) + r'\b', text)]

            return ','.join(found_attributes) if found_attributes else ""
        
        return ""
    

def find_pos_attributes(attributes, text,is_number=False):
    found_attributes = []
    if pd.notna(text):
        word = text.split()
        for i, word in enumerate(word):
            if is_number:
                if re.match(r'\d+', word):
                    found_attributes.append(i+1)
            elif word in attributes:                
                found_attributes.append(i+1)             
        return found_attributes

def attribute_strpos(attributes, text, label,is_number=False):
    if pd.notna(text):
        found_attributes = []
        for attribute in attributes:
            if attribute in text:
                start = text.find(attribute)
                end = start + len(attribute)
                found_attributes.append([start, end, label])
                
            elif is_number : 
                numbers = re.findall(r'\d+', text)
                for number in numbers : 
                    start = text.find(number)
                    end = start + len(number)
                    found_attributes.append([start, end, label])
        return found_attributes
    else:
        return []

    
def annotate(text, attributes):
    tags = []

    for label, values in attributes.items():
        
        if label =='numeros':
            attribute = attribute_strpos(values, text, label,True)
            
        else : 
            attribute = attribute_strpos(values, text, label)
            
        if attribute:
            tags.append(attribute)

    return tags

In [29]:

df['voirie'] = df.apply(lambda x: find_attribute(voirie, x['ADRESS']), axis=1)
df['numeros'] = df.apply(lambda x: find_attribute(numeros, x['ADRESS'],is_number=True), axis=1)
df['bruit'] = df.apply(lambda x: find_attribute(bruit, x['ADRESS']), axis=1)

df['pos_voirie'] = df.apply(lambda x: find_pos_attributes(voirie, x['ADRESS']), axis=1)
df['pos_numeros'] = df.apply(lambda x: find_pos_attributes(numeros, x['ADRESS'],is_number=True), axis=1)
df['pos_bruit'] = df.apply(lambda x: find_pos_attributes(bruit, x['ADRESS']), axis=1)

attributes = {
    'voirie': voirie,
    'numeros': numeros,
    'bruit': bruit,
}
df.dropna(subset="ADRESS")
df['label'] = df.apply(lambda x: annotate(x['ADRESS'], attributes), axis=1)

/tmp/ipykernel_2478178/758979128.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['voirie'] = df.apply(lambda x: find_attribute(voirie, x['ADRESS']), axis=1)
/tmp/ipykernel_2478178/758979128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['numeros'] = df.apply(lambda x: find_attribute(numeros, x['ADRESS'],is_number=True), axis=1)
/tmp/ipykernel_2478178/758979128.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

In [31]:
def extract_position(row):
    position = {
        'voirie': [],
        'numeros': [],
        'bruit':[]
    }
    
    for all_label in row : 
        for label in all_label:          
            start, end, label_type = label
            position[label_type].append(start)
        
        
    return position

#df_test = df[:10]
positions_data = df['label'].apply(extract_position)

In [32]:
# Rassembler les données pour chaque type de bruit
voirie_positions = [pos for row in positions_data for pos in row['voirie']]
numeros_positions = [pos for row in positions_data for pos in row['numeros']]
bruit_positions = [pos for row in positions_data for pos in row['bruit']]

# pd.DataFrame(voirie_positions).to_csv("./biais/pos_voirie.csv",sep=";")
# pd.DataFrame(numeros_positions).to_csv("./biais/pos_numeros.csv",sep=";")
# pd.DataFrame(bruit_positions).to_csv("./biais/pos_bruit.csv",sep=";")


In [35]:
df_long = pd.DataFrame({
    'Type': ['Numéros'] * len(numeros_positions) + ['Voirie'] * len(voirie_positions) + ['Bruit'] * len(bruit_positions),
    'Position': numeros_positions + voirie_positions + bruit_positions
})

# Créer le graphique à violons
fig = px.violin(df_long, y='Position', x='Type', box=True)


fig.update_layout(
    title="Répartition des Positions des Numéros, Voirie et Bruits dans les Adresses",
    yaxis_title="Position dans l'Adresse",
    xaxis_title="Type"
)

# Sauvegarder le graphique en tant qu'image
fig.write_image("images/violin_all_gendarmerie.png")

# Afficher le graphique
#fig.show()